<a href="https://colab.research.google.com/github/fatimatouda8-create/Exam_invig_2da/blob/main/last_one.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pulp
import pandas as pd
import numpy as np

# ==================== PARAMETERS FROM THE PROBLEM ====================
n = 39  # number of exams
T = 9   # exam period length (days)
l = 2   # max invigilations per day per professor
P = 4  # max invigilation days per professor


In [ ]:
# ==================== CREATE DATA FROM TABLE 1 ====================
exams_data = [
       (1, 1, 1, "Monday", "t1", "S1MAT100", 90, "PWD", "A", "Gala"),
            (2, 2, 1, "Monday", "t1", "M416", 30, "PWD", "3", "Beddani"),
            (3, 3, 1, "Monday", "t2", "M216", 42, "PWOD", "15", "Abassa"),
            (4, 4, 1, "Monday", "t2", "M311", 32, "PWOD", "3", "Djelloul"),
            (5, 5, 1, "Monday", "t2", "M511", 37, "PWOD", "5", "Hakiki"),
            (6, 6, 2, "Tuesday", "t3", "AN1", 90, "PWD", "A", "Belkhir"),
            (7, 7, 2, "Tuesday", "t3", "M473", 37, "PWOD", "5", "Ben Hamida"),
            (8, 8, 2, "Tuesday", "t4", "M212", 42, "PWOD", "15", "Belarbi"),
            (9, 9, 2, "Tuesday", "t4", "M461", 30, "PWOD", "3", "Menad"),
            (10, 10, 3, "Wednesday", "t1", "S1MAT104", 90, "PWD", "A", "Bahnes Z"),
            (11, 7, 3, "Wednesday", "t1", "M313", 32, "PWOD", "3", "Ben Hamida"),
            (12, 11, 3, "Wednesday", "t2", "M215", 42, "PWOD", "15", "Bouich"),
            (13, 12, 3, "Wednesday", "t2", "M415", 30, "PWOD", "3", "Bar"),
            (14, 13, 3, "Wednesday", "t2", "M514", 37, "PWOD", "5", "Mahi"),
            (15, 14, 4, "Thursday", "t3", "HISS1", 90, "PWD", "A", "Larabi"),
            (16, 15, 4, "Thursday", "t3", "M414", 30, "PWOD", "3", "Henine"),
            (17, 14, 4, "Thursday", "t3", "E472", 37, "PWD", "5", "Larabi"),
            (18, 16, 4, "Thursday", "t4", "I241", 42, "PWOD", "15", "Bel Aicha"),
            (19, 17, 4, "Thursday", "t4", "M312", 32, "PWOD", "3", "Snousi"),
            (20, 18, 5, "Saturday", "t1", "S1MAT101", 90, "PWOD", "A", "Messirdi"),
            (21, 17, 5, "Saturday", "t1", "M413", 30, "PWOD", "3", "Snousi"),
            (22, 19, 5, "Saturday", "t2", "M213", 42, "PWOD", "15", "Ben Barnou"),
            (23, 20, 5, "Saturday", "t2", "M315", 32, "PWOD", "3", "Bouzian"),
            (24, 21, 5, "Saturday", "t2", "M512", 37, "PWOD", "5", "Makreloufi"),
            (25, 12, 6, "Sunday", "t3", "M214", 42, "PWOD", "15", "Bar"),
            (26, 15, 6, "Sunday", "t3", "M314", 32, "PWOD", "3", "Henine"),
            (27, 22, 6, "Sunday", "t3", "M513", 37, "PWOD", "5", "Bahri"),
            (28, 16, 6, "Sunday", "t4", "TIC", 90, "PWOD", "A", "Bel Aicha"),
            (29, 20, 6, "Sunday", "t4", "M411", 30, "PWOD", "3", "Bouzian"),
            (30, 5, 7, "Tuesday", "t1", "M211", 42, "PWOD", "15", "Hakiki"),
            (31, 13, 7, "Tuesday", "t1", "M316", 32, "PWOD", "3", "Mahi"),
            (32, 9, 7, "Tuesday", "t1", "M561", 37, "PWOD", "5", "Menad"),
            (33, 23, 7, "Tuesday", "t2", "S1MAT103", 90, "PWD", "A", "Mokran"),
            (34, 22, 7, "Tuesday", "t2", "M412", 30, "PWOD", "3", "Bahri"),
            (35, 21, 8, "Wednesday", "t3", "S1MAT102", 90, "PWOD", "A", "Makreloufi"),
            (36, 11, 8, "Wednesday", "t4", "M317", 32, "PWOD", "5", "Bouich"),
            (37, 14, 8, "Wednesday", "t4", "E471", 30, "PWD", "3", "Larabi"),
            (38, 24, 9, "Thursday", "t1", "E271", 42, "PWOD", "15", "Aoun"),
            (39, 24, 9, "Thursday", "t2", "E371", 32, "PWOD", "3", "Aoun"),
]

# Create DataFrame
columns = ['i', 'j', 'Day', 'Day_Name', 'Time', 'Course', 'Students', 'TOI', 'Room', 'Teacher_Name']
df_exams = pd.DataFrame(exams_data, columns=columns)

# Calculate required invigilators Pi = ceil(Students/20)
df_exams['Required_Invigilators'] = np.ceil(df_exams['Students'] /20).astype(int)


In [ ]:
# ==================== SET DEFINITIONS ====================
E = list(range(1, n+1))  # Exams {1, ..., 39}
D = list(range(1, T+1))  # Days {1, ..., 9}

# Get all professor IDs from column j
all_professors = sorted(df_exams['j'].unique())

# Get teacher names mapping
teacher_names = {}
for exam in exams_data:
    teacher_id = exam[1]
    teacher_name = exam[9]
    teacher_names[teacher_id] = teacher_name

# Identify PWD and PWOD from TOI column
A = []  # PWD professors
S = []  # PWOD professors

prof_type = {}
for exam in exams_data:
    prof_id = exam[1]
    prof_toi = exam[7]  # TOI column
    if prof_toi == 'PWD' and prof_id not in A:
        A.append(prof_id)
        prof_type[prof_id] = 'PWD'
    elif prof_toi == 'PWOD' and prof_id not in S:
        S.append(prof_id)
        prof_type[prof_id] = 'PWOD'

A = sorted(set(A))
S = sorted(set(S))

print(f"PWD Professors (A): {A}")
print(f"PWOD Professors (S): {S}")
print(f"Total professors: {len(A) + len(S)}")

# Eo sets (exams at same time)
Eo_sets = [
    {1, 2}, {3, 4, 5}, {6, 7}, {8, 9}, {10, 11},
    {12, 13, 14}, {15, 16, 17}, {18, 19}, {20, 21},
    {22, 23, 24}, {25, 26, 27}, {28, 29}, {30, 31, 32},
    {33, 34}, {35, 36, 37}
]

# Ed sets (exams per day)
Ed = {}
for day in D:
    Ed[day] = df_exams[df_exams['Day'] == day]['i'].tolist()

# M(j) - exams taught by professor j
M = {}
for exam in exams_data:
    exam_id = exam[0]
    professor_id = exam[1]
    if professor_id not in M:
        M[professor_id] = []
    M[professor_id].append(exam_id)

# Calculate required PWOD invigilators for each exam
pwod_required_per_exam = {}
for i in E:
    exam_info = df_exams[df_exams['i'] == i].iloc[0]
    total_required = exam_info['Required_Invigilators']

    # Count how many PWD professors teach this exam
    pwd_count = 0
    prof_id = exam_info['j']
    if prof_id in A:
        pwd_count = 1

    # Required PWOD = Total required - PWD professors (who must invigilate)
    pwod_required = total_required - pwd_count
    pwod_required_per_exam[i] = max(0, pwod_required)


PWD Professors (A): [1, 2, 6, 10, 14, 23]
PWOD Professors (S): [3, 4, 5, 7, 8, 9, 11, 12, 13, 15, 16, 17, 18, 19, 20, 21, 22, 24]
Total professors: 24


In [ ]:
# ==================== CREATE AND SOLVE MODEL ====================
print("\nCreating optimization model...")
prob = pulp.LpProblem('Invigilation_Scheduling', pulp.LpMinimize)

# Decision variables xij only for PWOD professors
x = pulp.LpVariable.dicts('x',
                         [(i, j) for i in E for j in S],
                         lowBound=0, upBound=1, cat='Binary')

# y variables only for PWOD professors
y = pulp.LpVariable.dicts('y',
                         [(j, d) for j in S for d in D],
                         lowBound=0, upBound=1, cat='Binary')

# Lmax variable for PWOD workload balance
Lmax = pulp.LpVariable('Lmax', lowBound=0, cat='Continuous')

# Objective: minimize maximum workload among PWOD
prob += Lmax



Creating optimization model...


In [ ]:
# ==================== ADD CONSTRAINTS ====================
print("Adding constraints...")

# 1. Clash-free requirement for PWOD
for conflict_set in Eo_sets:
    for j in S:
        prob += pulp.lpSum(x[(i, j)] for i in conflict_set) <= 1

# 2. Required PWOD invigilators per exam
for i in E:
    required_pwod = pwod_required_per_exam[i]
    if required_pwod > 0:
        prob += pulp.lpSum(x[(i, j)] for j in S) == required_pwod
    else:
        prob += pulp.lpSum(x[(i, j)] for j in S) == 0

# 3. Daily limit for PWOD professors
for d in D:
    for j in S:
        prob += pulp.lpSum(x[(i, j)] for i in Ed[d]) <= l

# 4. Each PWOD professor must invigilate their own module
for j in S:
    if j in M:
        for exam_id in M[j]:
            if j in S:
                prob += x[(exam_id, j)] == 1

# 5. Day-limit constraints for PWOD
for d in D:
    for j in S:
        prob += pulp.lpSum(x[(i, j)] for i in Ed[d]) <= l * y[(j, d)]

for j in S:
    prob += pulp.lpSum(y[(j, d)] for d in D) <= P

# 6. Workload balance for PWOD
for j in S:
    Lj = pulp.lpSum(x[(i, j)] for i in E)
    prob += Lj <= Lmax


Adding constraints...


In [ ]:
# ==================== SOLVE THE MODEL ====================
print("\nSolving the model...")
solver = pulp.PULP_CBC_CMD(msg=False, timeLimit=60)
prob.solve(solver)

print(f"\nSolution Status: {pulp.LpStatus[prob.status]}")
if prob.status == pulp.LpStatusOptimal:
    print(f"✓ Optimal solution found!")
    print(f"Objective Value (Lmax for PWOD): {pulp.value(Lmax):.0f}")
else:
    print(f"✗ Solution status: {prob.status}")
    exit()


Solving the model...

Solution Status: Optimal
✓ Optimal solution found!
Objective Value (Lmax for PWOD): 6


In [ ]:
# ==================== COLLECT RESULTS ====================
print("\n" + "="*100)
print("COMPLETE INVIGILATION ASSIGNMENTS")
print("="*100)

# 1. Get PWOD assignments from the model
assignments_pwod = []
for i in E:
    for j in S:
        if pulp.value(x[(i, j)]) == 1:
            exam_info = df_exams[df_exams['i'] == i].iloc[0]
            assignments_pwod.append({
                'Exam_ID': i,
                'Professor_ID': j,
                'Professor_Name': teacher_names[j],
                'Type': 'PWOD',
                'Day': exam_info['Day'],
                'Day_Name': exam_info['Day_Name'],
                'Time_Slot': exam_info['Time'],
                'Course': exam_info['Course'],
                'Room': exam_info['Room'],
                'Students': exam_info['Students'],
                'Required_Invigilators': exam_info['Required_Invigilators']
            })

df_assignments_pwod = pd.DataFrame(assignments_pwod)

# 2. Add PWD assignments (they invigilate their own exams)
assignments_pwd = []
for i in E:
    exam_info = df_exams[df_exams['i'] == i].iloc[0]
    prof_id = exam_info['j']
    if prof_id in A:
        assignments_pwd.append({
            'Exam_ID': i,
            'Professor_ID': prof_id,
            'Professor_Name': teacher_names[prof_id],
            'Type': 'PWD',
            'Day': exam_info['Day'],
            'Day_Name': exam_info['Day_Name'],
            'Time_Slot': exam_info['Time'],
            'Course': exam_info['Course'],
            'Room': exam_info['Room'],
            'Students': exam_info['Students'],
            'Required_Invigilators': exam_info['Required_Invigilators']
        })

df_assignments_pwd = pd.DataFrame(assignments_pwd)

# 3. Combine all assignments
df_all_assignments = pd.concat([df_assignments_pwd, df_assignments_pwod], ignore_index=True)

# 4. Calculate professor workload
professor_load = {}
all_profs = sorted(set(A + S))

for prof_id in all_profs:
    if prof_id in A:
        pwd_count = len([i for i in E if df_exams[df_exams['i'] == i].iloc[0]['j'] == prof_id])
    else:
        pwd_count = 0

    if prof_id in S:
        pwod_count = len(df_assignments_pwod[df_assignments_pwod['Professor_ID'] == prof_id])
    else:
        pwod_count = 0

    total = pwd_count + pwod_count

    professor_load[prof_id] = {
        'name': teacher_names[prof_id],
        'type': 'PWD' if prof_id in A else 'PWOD',
        'total': total,
        'pwd_assignments': pwd_count,
        'pwod_assignments': pwod_count
    }

# 5. Calculate PWOD professor invigilation days
professor_days = {}
for j in S:
    days = sum(pulp.value(y[(j, d)]) for d in D)
    professor_days[j] = days



COMPLETE INVIGILATION ASSIGNMENTS


In [ ]:
# ==================== SAVE RESULTS TO CSV FILES ====================
print("\n\nSaving results to CSV files...")

# Define time_details dictionary
time_details = {
    't1': '08:00 - 09:30',
    't2': '10:00 - 11:30',
    't3': '13:00 - 14:30',
    't4': '15:00 - 16:30'
}

# 1. Save professor summary
prof_summary_data = []
for prof_id in sorted(all_profs):
    prof_info = professor_load[prof_id]
    prof_summary_data.append({
        'Professor_ID': prof_id,
        'Name': prof_info['name'],
        'Type': prof_info['type'],
        'Total_Invigilations': prof_info['total'],
        'PWD_Exams': prof_info['pwd_assignments'],
        'PWOD_Assignments': prof_info['pwod_assignments'],
        'Invigilation_Days': professor_days.get(prof_id, 'N/A') if prof_id in S else 'N/A'
    })

df_prof_summary = pd.DataFrame(prof_summary_data)
df_prof_summary.to_csv('professor_summary.csv', index=False)

# 2. Save schedule table
schedule_data = []
for day in sorted(D):
    day_name = df_exams[df_exams['Day'] == day]['Day_Name'].iloc[0]
    day_exams = sorted(df_exams[df_exams['Day'] == day]['i'].unique())

    for exam_id in day_exams:
        exam_info = df_exams[df_exams['i'] == exam_id].iloc[0]
        exam_assignments = df_all_assignments[df_all_assignments['Exam_ID'] == exam_id]

        professors = []
        for _, row in exam_assignments.iterrows():
            professors.append(f"{row['Professor_Name']} ({row['Type']})")

        schedule_data.append({
            'Day': day,
            'Day_Name': day_name,
            'Time_Slot': exam_info['Time'],
            'Time_Range': time_details[exam_info['Time']],
            'Room': exam_info['Room'],
            'Exam_ID': exam_id,
            'Course': exam_info['Course'],
            'Students': exam_info['Students'],
            'Required_Invigilators': exam_info['Required_Invigilators'],
            'Assigned_Professors': ', '.join(professors)
        })

df_schedule = pd.DataFrame(schedule_data)
df_schedule.to_csv('exam_schedule.csv', index=False)




# 6. Create a summary Excel file with multiple sheets
print("\nCreating comprehensive Excel file...")
with pd.ExcelWriter('invigilation_complete_report.xlsx', engine='openpyxl') as writer:


    # Sheet 1: Professor summary
    df_prof_summary.to_excel(writer, sheet_name='Professor_Summary', index=False)

    # Sheet 2: Exam schedule
    df_schedule.to_excel(writer, sheet_name='Exam_Schedule', index=False)


print("\nFiles saved:")
print("1. professor_summary.csv - Professor workload summary")
print("2. exam_schedule.csv - Complete exam schedule")
print("3. invigilation_complete_report.xlsx - Comprehensive report in Excel")
print("\n" + "="*80)
print("SIMULATION COMPLETE")
print("="*80)




Saving results to CSV files...

Creating comprehensive Excel file...

Files saved:
1. professor_summary.csv - Professor workload summary
2. exam_schedule.csv - Complete exam schedule
3. invigilation_complete_report.xlsx - Comprehensive report in Excel

SIMULATION COMPLETE
